In [1]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd
import os


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random

def generate_negative_edges(df, num_negatives, negative_score=-1, exclude_pairs=None):
    """
    Efficiently generate up to `num_negatives` negative edges not in `exclude_pairs` or df.
    """
    cdr3_pool = df['cdr3'].unique()
    epitope_pool = df['epitope'].unique()

    existing_pairs = set(tuple(x) for x in df[['cdr3', 'epitope']].values)
    if exclude_pairs:
        existing_pairs |= set(exclude_pairs)

    negatives = []
    tried = set()

    while len(negatives) < num_negatives and len(tried) < len(cdr3_pool) * len(epitope_pool):
        cdr3 = random.choice(cdr3_pool)
        epitope = random.choice(epitope_pool)
        pair = (cdr3, epitope)

        if pair not in existing_pairs and pair not in tried:
            tried.add(pair)
            negatives.append({
                'cdr3': cdr3,
                'epitope': epitope,
                'score': negative_score,
                'source': 'NEGATIVE',
                'label': 0,
                # 'cdr3_truncated': truncate(cdr3)
            })

    return pd.DataFrame(negatives)


# def truncate(seq, width=4):
#     mid = (len(seq) + 1) // 2
#     return seq[max(0, mid - width):min(len(seq), mid + width)]


def sample_data(df, source=None, min_score=None, max_score=None, num_negatives=0, 
                negative_score=-1, exclude_sources=None, exclude_mode='pair', n=1000):
    """
    Filter and sample from data with support for flexible exclusion and negative generation.
    """
    filtered_df = df.copy()

    # Filter by source
    if source:
        source = [source] if isinstance(source, str) else source
        filtered_df = filtered_df[filtered_df['source'].isin(source)]

    # Score filtering
    if min_score is not None:
        filtered_df = filtered_df[filtered_df['score'] >= min_score]
    if max_score is not None:
        filtered_df = filtered_df[filtered_df['score'] <= max_score]

    # Exclude logic
    exclude_pairs = []
    if exclude_sources:
        exclude_sources = [exclude_sources] if isinstance(exclude_sources, str) else exclude_sources
        exclude_df = df[df['source'].isin(exclude_sources)][['cdr3', 'epitope']].dropna()

        if exclude_mode == 'pair':
            exclude_pairs = set(tuple(x) for x in exclude_df.values)
            filtered_df = filtered_df[~filtered_df[['cdr3', 'epitope']].apply(tuple, axis=1).isin(exclude_pairs)]

        elif exclude_mode == 'epitope':
            excluded_epitopes = exclude_df['epitope'].unique()
            filtered_df = filtered_df[~filtered_df['epitope'].isin(excluded_epitopes)]

        elif exclude_mode == 'tcr':
            excluded_cdr3s = exclude_df['cdr3'].unique()
            filtered_df = filtered_df[~filtered_df['cdr3'].isin(excluded_cdr3s)]
            
        elif exclude_mode == 'both':
            excluded_epitopes = exclude_df['epitope'].unique()
            excluded_cdr3s = exclude_df['cdr3'].unique()
            filtered_df = filtered_df[~filtered_df['cdr3'].isin(excluded_cdr3s)]
            filtered_df = filtered_df[~filtered_df['epitope'].isin(excluded_epitopes)]

    # Assign positive labels and truncate
    filtered_df = filtered_df.copy()
    filtered_df['label'] = 1
    # filtered_df['cdr3_truncated'] = filtered_df['cdr3'].apply(truncate)
    filtered_df = filtered_df.drop_duplicates(subset=['cdr3', 'epitope'])
    filtered_df = filtered_df.sample(n=min(n, len(filtered_df)), random_state=42)

    # filtered_df = filtered_df.sample(n=n, random_state=42)

    # Generate negatives
    if num_negatives == -1:
        num_negatives = len(filtered_df)

    if num_negatives > 0:
        negative_df = generate_negative_edges(
            df,
            num_negatives,
            negative_score,
            exclude_pairs=exclude_pairs if exclude_mode == 'pair' else None
        )
        filtered_df = pd.concat([filtered_df, negative_df], ignore_index=True)

    
    return filtered_df


In [3]:
def create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=1000, distance_threshold = 0.2, use_mst=True, enrich=False):

    data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, label_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    print("Done indexing")

    data_hetero, labels = gnn_utils.build_hetero_graph(
                            data,
                            chephy_matrix,
                            sequences_list,
                            epitopes,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    return data_hetero, data


In [4]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [5]:
data_combined = pd.read_csv('/home/dsi/chenbis/repos/sol_lab/files/combined_data.csv')
sampled = sample_data(
    data_combined,
    source='IEDB',
    n=45000,
    
)


In [6]:
from sklearn.model_selection import train_test_split

def split_disjoint_cdr3_epitope(df, test_size=0.2, random_state=42):
    # Step 1: Get unique CDR3s and epitopes
    unique_cdr3 = df['cdr3'].unique()
    unique_epitopes = df['epitope'].unique()

    # Step 2: Split each into train/test
    cdr3_train, cdr3_test = train_test_split(unique_cdr3, test_size=test_size, random_state=random_state)
    epitope_train, epitope_test = train_test_split(unique_epitopes, test_size=test_size, random_state=random_state)

    # Step 3: Assign each row to train/test if both CDR3 and epitope are in the same split
    train_mask = df['cdr3'].isin(cdr3_train) & df['epitope'].isin(epitope_train)
    test_mask  = df['cdr3'].isin(cdr3_test) & df['epitope'].isin(epitope_test)

    train_df = df[train_mask].reset_index(drop=True)
    test_df = df[test_mask].reset_index(drop=True)
    discarded = len(df) - len(train_df) - len(test_df)
    print(f"{discarded} rows discarded due to cross-split overlap.")

    return train_df, test_df


In [7]:

data_train, data_test = split_disjoint_cdr3_epitope(sampled, test_size=0.35, random_state=42)
# data_train, data_test = train_test_split(sampled, test_size=0.2)
len(data_train), len(data_test)

21079 rows discarded due to cross-split overlap.


(17437, 6484)

In [8]:
len(set(data_train["cdr3"])), len(set(data_test["cdr3"]))

(17252, 6390)

In [9]:
len(set(data_train["epitope"])), len(set(data_test["epitope"]))

(390, 185)

In [10]:
# Check for overlapping TCRs
tcr_overlap = set(data_train['cdr3']).intersection(set(data_test['cdr3']))
print(f"Number of overlapping TCRs: {len(tcr_overlap)}")

# Check for overlapping epitopes
epitope_overlap = set(data_train['epitope']).intersection(set(data_test['epitope']))
print(f"Number of overlapping epitopes: {len(epitope_overlap)}")

# Check for overlapping pairs
pair_overlap = set(zip(data_train['cdr3'], data_train['epitope'])).intersection(
    set(zip(data_test['cdr3'], data_test['epitope']))
)
print(f"Number of overlapping pairs: {len(pair_overlap)}")

Number of overlapping TCRs: 0
Number of overlapping epitopes: 0
Number of overlapping pairs: 0


In [11]:
data_train = pd.concat([data_train, generate_negative_edges(data_train, len(data_train))], ignore_index=True)
data_test = pd.concat([data_test, generate_negative_edges(data_test, len(data_test))], ignore_index=True)


In [12]:
# input_file = "/home/dsi/chenbis/repos/sol_lab/data/train.tsv"
# data_train = run.load_dataframe(input_file)
# data_positive = data[data["Binding"]==1]

label_header = "epitope"
cdr3_header = "cdr3"

n = len(data_train)  # Use the length of the training data for sampling
distance_threshold=0.05
use_mst=False
enrich=True

data_hetero_train, data_train = create_hetero_data(data_train, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)


sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
skip mst
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [13]:
# input_file = "/home/dsi/chenbis/repos/sol_lab/data/test.tsv"
# data = run.load_dataframe(input_file)
# data_positive = data[data["Binding"]==1]

label_header = "epitope"
cdr3_header = "cdr3"
n = len(data_test)  # Use the length of the test data for sampling

distance_threshold=0.05
use_mst=False
enrich=True

data_hetero_test, data_test = create_hetero_data(data_test, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)


sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
skip mst
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [14]:
model = GNN_model.HeteroTCR(data_hetero_train.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_train = data_hetero_train.to(device)
data_hetero_test = data_hetero_test.to(device)

In [15]:
# Create edge label indices for training data
train_edge_pairs = data_train[['tcr_index', 'epitope_index']].values
train_edge_label_index = torch.tensor(train_edge_pairs.T)

# Create edge label indices for test data
test_edge_pairs = data_test[['tcr_index', 'epitope_index']].values
test_edge_label_index = torch.tensor(test_edge_pairs.T)

# Create labels
y_train = data_train['label'].values
y_test = data_test['label'].values

In [16]:
torch.cuda.empty_cache()

In [17]:
import os
import torch
import pandas as pd


save_model_path = f'/home/dsi/chenbis/repos/sol_lab/output/models/{run.get_time()}'
os.makedirs(save_model_path, exist_ok=True)
NUM_EPOCHS = 1000

# Track performance
epoch = []
loss = []
acc = []
auc_roc = []
val_loss = []
val_acc = []
val_auc_roc = []

# Best model tracking
best_val_auc = -float('inf')
best_val_loss = float('inf')
best_auc_model_state = None
best_loss_model_state = None
best_auc_epoch = 0
best_loss_epoch = 0

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0)

for ep in range(1, NUM_EPOCHS + 1):
    train_loss, train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC = GNN_model.train(
        model, optimizer,
        data_hetero_train, data_hetero_test,
        train_edge_label_index, test_edge_label_index,
        y_train, y_test,
        device
    )

    print(
        'Train epoch: {} - loss: {:.4f} - binary_accuracy: {:.4f} - ROCAUC: {:.4f} - val_loss: {:.4f} - val_binary_accuracy: {:.4f} - val_ROCAUC: {:.4f}'.format(
            ep, train_loss.item(), train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC))

    # Save best AUC model
    if val_ROCAUC > best_val_auc:
        best_val_auc = val_ROCAUC
        best_auc_model_state = model.state_dict()
        best_auc_epoch = ep

    # Save best loss model
    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        best_loss_model_state = model.state_dict()
        best_loss_epoch = ep

    # Store metrics
    epoch.append(ep)
    loss.append(train_loss.item())
    acc.append(train_binary_accuracy.item())
    auc_roc.append(train_ROCAUC.item())
    val_loss.append(valid_loss.item())
    val_acc.append(val_binary_accuracy.item())
    val_auc_roc.append(val_ROCAUC.item())

# Save training history
root_history = '/home/dsi/chenbis/repos/sol_lab/output'
dfhistory = {
    'epoch': epoch,
    'loss': loss, 'acc': acc, 'auc_roc': auc_roc,
    'val_loss': val_loss, 'val_acc': val_acc, 'val_auc_roc': val_auc_roc
}
df = pd.DataFrame(dfhistory)
df.to_csv(os.path.join(root_history, "history.csv"), header=True, index=False)

# Save best models to disk
torch.save(best_auc_model_state, os.path.join(save_model_path, f"max_AUC_HeteroTCR_epoch{best_auc_epoch:03d}_AUC{best_val_auc:.6f}.pth"))
torch.save(best_loss_model_state, os.path.join(save_model_path, f"minloss_AUC_HeteroTCR_epoch{best_loss_epoch:03d}_LOSS{best_val_loss:.6f}.pth"))

print("Best AUC model saved from epoch:", best_auc_epoch, "with AUC:", best_val_auc)
print("Best loss model saved from epoch:", best_loss_epoch, "with loss:", best_val_loss)


Train epoch: 1 - loss: 0.6937 - binary_accuracy: 0.5014 - ROCAUC: 0.4980 - val_loss: 0.7145 - val_binary_accuracy: 0.5000 - val_ROCAUC: 0.5120
Train epoch: 2 - loss: 0.7165 - binary_accuracy: 0.5000 - ROCAUC: 0.5177 - val_loss: 0.6934 - val_binary_accuracy: 0.5000 - val_ROCAUC: 0.5296
Train epoch: 3 - loss: 0.6931 - binary_accuracy: 0.5000 - ROCAUC: 0.5528 - val_loss: 0.6951 - val_binary_accuracy: 0.5000 - val_ROCAUC: 0.5272
Train epoch: 4 - loss: 0.6952 - binary_accuracy: 0.5000 - ROCAUC: 0.5507 - val_loss: 0.6941 - val_binary_accuracy: 0.5000 - val_ROCAUC: 0.5393
Train epoch: 5 - loss: 0.6941 - binary_accuracy: 0.5000 - ROCAUC: 0.5617 - val_loss: 0.6923 - val_binary_accuracy: 0.5000 - val_ROCAUC: 0.5754
Train epoch: 6 - loss: 0.6921 - binary_accuracy: 0.5180 - ROCAUC: 0.5873 - val_loss: 0.6924 - val_binary_accuracy: 0.5000 - val_ROCAUC: 0.5628
Train epoch: 7 - loss: 0.6921 - binary_accuracy: 0.5007 - ROCAUC: 0.5762 - val_loss: 0.6921 - val_binary_accuracy: 0.5000 - val_ROCAUC: 0.5850

# Independet Test

In [18]:
# save_model_path = "/home/dsi/chenbis/repos/sol_lab/output/models/20250617_093758"
data_eval = sample_data(
    data_combined,
    source='VDJDB',
    exclude_sources='IEDB', 
    exclude_mode="both" # Exclude IEDB sources
    
)

In [19]:

data_eval = pd.concat([data_eval, generate_negative_edges(data_eval, len(data_eval))], ignore_index=True)


label_header = "epitope"
cdr3_header = "cdr3"

n = len(data_eval)  # Use the full dataset for evaluation
distance_threshold=0.05
use_mst=False
enrich=True

data_hetero_eval, data_eval = create_hetero_data(data_eval, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)


sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
skip mst
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [20]:

# Create edge label indices for test data
eval_edge_pairs = data_eval[['tcr_index', 'epitope_index']].values
eval_edge_label_index = torch.tensor(eval_edge_pairs.T)

# Create labels
y_eval = np.where(data_eval['source'] == 'NEGATIVE', 0, 1)

In [21]:
model = GNN_model.HeteroTCR(data_hetero_eval.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_eval = data_hetero_eval.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(data_hetero_eval.x_dict, data_hetero_eval.edge_index_dict, eval_edge_label_index)


model_dir = save_model_path
val_model_path = save_model_path
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, data_hetero_eval, eval_edge_label_index, y_eval)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

ACC: 0.6010
AUC: 0.6796


/tmp/ipykernel_1612443/2741821409.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH))


# Benchmark

In [13]:
import os

def get_file_from_latest_folder(base_path, prefix='max'):
    # Get list of subfolders
    subfolders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]
    
    # Sort and get the latest one (assuming names are sortable)
    latest_folder = max(subfolders, default=None)

    if not latest_folder:
        return None

    latest_folder_path = os.path.join(base_path, latest_folder)

    # Find file starting with the given prefix
    for filename in os.listdir(latest_folder_path):
        if filename.startswith(prefix):
            return os.path.join(latest_folder_path, filename)

    return None  # If no file with the prefix is found

# Example usage
base_directory = '/home/dsi/chenbis/repos/sol_lab/output/models'
save_model_path = get_file_from_latest_folder(base_directory, prefix='max')
# min_file = get_file_from_latest_folder(base_directory, prefix='min')


In [14]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd


def create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=1000, distance_threshold = 0.2, use_mst=True, enrich=False):

    data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, label_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    print("Done indexing")

    data_hetero, labels = gnn_utils.build_hetero_graph(
                            data,
                            chephy_matrix,
                            sequences_list,
                            epitopes,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    return data_hetero, data


atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



In [15]:
def load_your_trained_model(model_path, metadata):
    model = GNN_model.HeteroTCR(metadata=metadata)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

def prediction_func(df_input, model_path=None, **kwargs):
    df_output = df_input.copy()

    cdr3_header = "CDR3_beta"
    label_header = "Epitope"

    n = len(df_input)
    distance_threshold=0.05
    use_mst=False
    enrich=False

    hetero, data = create_hetero_data(df_input, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)
    print(hetero)
    # Create edge label indices for test data
    edge_pairs = data[['tcr_index', 'epitope_index']].values
    edge_label_index = torch.tensor(edge_pairs.T)

    # Load trained model
    model = load_your_trained_model(save_model_path, hetero.metadata())
    model = model.to(device)
    # Prepare edge_label_index
    edge_label_index = hetero['tcr', 'binds', 'epitope'].edge_index

    # Run prediction
    with torch.no_grad():
        out = model(hetero.x_dict, hetero.edge_index_dict, edge_label_index)
        scores = out.cpu().numpy()

    df_output["Score"] = scores


    # # PATH = os.path.join(val_model_path, file)
    # y_eval = np.where(df_output['Label'] == 0, 0, 1)
    # # print(torch.tensor(y_eval).float())
    # test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, hetero, edge_label_index, y_eval)
    # print("ACC: {:.4f}".format(test_binary_accuracy))
    # print("AUC: {:.4f}".format(test_ROCAUC))


    return df_output


In [16]:
from tcr_benchmark.eval.allTests import NAME_2_TEST
from tcr_benchmark.pp.datasets import download_datasets

predictor_name = 'hetero_gnn'
dataset = 'viral'
download_datasets(dataset)
test = NAME_2_TEST[dataset](None)

config = {'model_path': '/home/dsi/chenbis/repos/sol_lab/output/models20250609_191843/minloss_AUC_HeteroTCR_epoch294_LOSS0.631632.pth'}  # point to your trained weights


results = test.run_tests(prediction_func, predictor_name, config)
# results.head()


100%|██████████| 3/3 [00:00<00:00, 2955.12it/s]

sequence_vectors done on CPU
pdist done on CPU
normalized done on CPU
Done indexing
skip mst
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!
HeteroData(
  tcr={ x=[626, 55] },
  epitope={ x=[14, 55] },
  (tcr, binds, epitope)={ edge_index=[2, 8932] },
  (epitope, rev_binds, tcr)={ edge_index=[2, 8932] },
  (tcr, similar, tcr)={ edge_index=[2, 418] }
)


/home/dsi/chenbis/repos/sol_lab/pycode/tcr_benchmark/eval/metrics.py:75: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * recalls * precisions / (recalls + precisions)


In [17]:
results[(results['Metric']=='AUC')
                & (results['Group']=='full_data')    
            ]

,Method,Dataset,Group,Support,Metric_Type,Metric,Value
75,hetero_gnn,Viral,full_data,8932,TTP,AUC,0.522357
